# DroneCamp Vision: treino piloto na GPU do Kaggle

Este notebook treina (ou continua) o piloto a partir de um pacote gerado no PC com `kaggle-package`.
Guia completo: `sistema_ia/docs/treino_kaggle.md` no repositório.

Antes de rodar:
1. **Settings → Accelerator: GPU T4 x1.** A P100 não serve: o PyTorch atual não roda na arquitetura dela.
2. **Settings → Internet: On** (clona o repositório e baixa os pesos oficiais; exige telefone verificado na conta).
3. **Add Input:** o Dataset **privado** com o `.zip` do pacote. Para continuar uma sessão anterior, adicione também a saída dela.
4. Rode com **Save Version → Save & Run All (Commit)**: a execução segue em segundo plano por até 12 h e a pasta `runs/` vira a saída da versão.

O resultado é uma pasta `runs/pilot_train_*` igual à do PC. Ela ainda precisa do `compare-models` contra o modelo de referência antes de qualquer adoção.

In [ ]:
# Ajuste só esta célula.
REPO_URL = "https://github.com/CFSJCODE/DroneCamp_Vision.git"
BRANCH = "main"
PACKAGE_DIR = "/kaggle/input"   # pasta do Dataset com o pacote (o package.json é procurado dentro dela)
CONTINUE_FROM = None             # ex.: "/kaggle/input/<saida-anterior>/runs/pilot_train_.../fit/weights/resume.pt"
MAX_HOURS = 11.0                 # para com segurança antes do limite de 12 h da sessão
EXTRA_ARGS = []                  # ex.: ["--batch", "8"] se faltar memória na GPU

In [ ]:
import json, os, pathlib, shutil, subprocess, sys

print(sys.version)
subprocess.run(["nvidia-smi"], check=False)

## Código e ambiente

O pacote exige Python 3.12 ou 3.13. Se o Python do Kaggle for de outra versão, a célula cria um ambiente 3.12 em `/tmp/venv` com o `uv` e instala o PyTorch com CUDA. O ambiente fica fora de `/kaggle/working` para não entrar na saída.

In [ ]:
WORK = pathlib.Path("/kaggle/working")
REPO = pathlib.Path("/tmp/DroneCamp_Vision")
SI = REPO / "sistema_ia"
if not REPO.exists():
    # Sem Git LFS: os pesos do repositório não são necessários; os oficiais são baixados pela Ultralytics.
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True,
                   env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], check=True)
# Sem LFS, os .pt do repositório são ponteiros de texto; apagados, a Ultralytics baixa os pesos oficiais.
for pointer in (SI / "models").rglob("*.pt"):
    if pointer.stat().st_size < 1024 and pointer.read_bytes().startswith(b"version https://git-lfs"):
        pointer.unlink()
# runs/ do clone aponta para /kaggle/working/runs: é o que o Kaggle salva como saída.
(WORK / "runs").mkdir(exist_ok=True)
if (SI / "runs").is_symlink() or (SI / "runs").is_file():
    (SI / "runs").unlink()
elif (SI / "runs").exists():
    shutil.rmtree(SI / "runs")
(SI / "runs").symlink_to(WORK / "runs")

if (3, 12) <= sys.version_info[:2] < (3, 14):  # versões aceitas pelo pyproject
    PY = sys.executable
    subprocess.run([PY, "-m", "pip", "install", "-q", "-e", str(SI)], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    UV = [sys.executable, "-m", "uv"]
    subprocess.run(UV + ["venv", "-q", "--python", "3.12", "/tmp/venv"], check=True)
    PY = "/tmp/venv/bin/python"
    # Tenta as rodas CUDA da mais antiga para a mais nova (a mais antiga exige o driver menos recente).
    for cuda in ("cu126", "cu128", "cu130"):
        if subprocess.run(UV + ["pip", "install", "-q", "--python", PY, "torch", "torchvision",
                                "--index-url", f"https://download.pytorch.org/whl/{cuda}"]).returncode == 0:
            break
    subprocess.run(UV + ["pip", "install", "-q", "--python", PY, "-e", str(SI)], check=True)

check = subprocess.run([PY, "-c", (
    "import torch, ultralytics; ok = torch.cuda.is_available(); "
    "print(torch.__version__, ultralytics.__version__, ok, torch.cuda.get_device_name(0) if ok else '-'); "
    "cap = torch.cuda.get_device_capability(0) if ok else (0, 0); "
    "raise SystemExit(0 if ok and cap >= (7, 0) else 3)")])
if check.returncode:
    raise SystemExit("Sem GPU compatível: escolha GPU T4 x1 em Settings → Accelerator.")

## Pacote

`kaggle-train` confere o SHA-256 de todos os arquivos antes de treinar. Esta célula só mostra o que foi encontrado (e descompacta o `.zip`, se o Kaggle não o tiver descompactado).


In [ ]:
import zipfile

manifests = sorted(pathlib.Path(PACKAGE_DIR).rglob("package.json"))
if not manifests:
    # Se o Kaggle manteve o .zip sem descompactar, extrai numa pasta temporária.
    archives = sorted(pathlib.Path(PACKAGE_DIR).rglob("dronecamp_kaggle_*.zip"))
    if len(archives) == 1:
        target = pathlib.Path("/tmp/pacote")
        shutil.rmtree(target, ignore_errors=True)
        zipfile.ZipFile(archives[0]).extractall(target)
        PACKAGE_DIR = str(target)
        manifests = sorted(target.rglob("package.json"))
print("\n".join(map(str, manifests)) or "Nenhum package.json encontrado: adicione o Dataset do pacote.")
if len(manifests) == 1:
    info = json.loads(manifests[0].read_text(encoding="utf-8"))
    print(json.dumps({k: info.get(k) for k in ("mode", "created_at", "git_commit", "source", "checkpoint", "counts")},
                     ensure_ascii=False, indent=2))


## Treino

Com `MAX_HOURS`, o treino para depois da primeira época que passar desse tempo e grava `fit/weights/resume.pt`.
Para continuar, adicione a saída desta versão como Input e aponte `CONTINUE_FROM` para esse arquivo.

In [ ]:
command = [PY, "-m", "dronecamp_ia", "kaggle-train", "--package", PACKAGE_DIR, "--device", "0",
           "--max-hours", str(MAX_HOURS)] + (["--checkpoint", CONTINUE_FROM] if CONTINUE_FROM else []) + EXTRA_ARGS
print(" ".join(command))
process = subprocess.Popen(command, cwd=SI, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
code = process.wait()
print("código de saída:", code)
if code:
    # Falha visível na versão do Kaggle (senão o "Save & Run All" aparece como sucesso).
    raise SystemExit(f"kaggle-train falhou com código {code}; veja o log acima.")

## Resultado

In [ ]:
runs = sorted((WORK / "runs").glob("pilot_train_*"))
if runs:
    run = runs[-1]
    summary = json.loads((run / "summary.json").read_text(encoding="utf-8"))
    print(run.name, "→", summary.get("state"), summary.get("reasons"))
    results = run / "fit" / "results.csv"
    if results.is_file():
        print("".join(results.read_text(encoding="utf-8").splitlines(keepends=True)[-5:]))
    for weight in sorted((run / "fit" / "weights").glob("*.pt")):
        print(weight.name, round(weight.stat().st_size / 1e6, 1), "MB")

## Levar o resultado para o PC

1. Baixe a pasta `runs/pilot_train_*` da aba **Output** da versão.
2. Copie para `sistema_ia\runs\` no PC.
3. Compare com o modelo de referência por classe (`compare-models`) antes de qualquer adoção.